# Validation JSON ↔ TTL 
Objectif : vérifier rapidement qu'aucune information *task/task category* n'est perdue entre le JSON et les TTL.

In [1]:
from pathlib import Path
import json
import re
import sys
from collections import Counter
from rdflib import Graph, Namespace
from rdflib.namespace import DCTERMS

def find_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "ontology").exists():
            return candidate
    return current

ROOT = find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.parsers.huggingFace_new_parser import parse

JSON_PATH = ROOT / "case-study" / "data" / "input" / "datasets_new_new.json"
TTL_DIRS =[ ROOT / "artifacts" / "kg" / "huggingface_new_new"]

MLUO = Namespace("http://example.org/mluo/ontology#")
MLUO_TH = Namespace("http://example.org/mluo/thesaurus#")
DATA_NS = Namespace("http://example.org/mluo/data#")

print({"root": str(ROOT), "json_exists": JSON_PATH.exists(), "ttl_dirs": [str(d) for d in TTL_DIRS]})

{'root': 'C:\\Users\\sunburst-user\\Documents\\datalens', 'json_exists': True, 'ttl_dirs': ['C:\\Users\\sunburst-user\\Documents\\datalens\\artifacts\\kg\\huggingface_new_new']}


In [2]:
def norm(x):
    return re.sub(r"[^a-z0-9]+", "-", str(x).strip().lower()).strip("-")

def as_list(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]

with open(JSON_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

json_rows = [obj for obj in raw if isinstance(obj, dict)]
parsed = [parse(obj) for obj in json_rows]
print({"json_raw": len(raw), "json_rows": len(json_rows), "json_parsed": len(parsed)})

{'json_raw': 901045, 'json_rows': 901045, 'json_parsed': 901045}


In [3]:
def json_stats(parsed_rows):
    task_counter = Counter()
    cat_counter = Counter()
    by_dataset = {}

    for row in parsed_rows:
        ds = row.get("dct_identifier")
        if not ds:
            continue

        usage = row.get("hasUsage", {}) or {}
        tasks_set = {norm(t) for t in as_list(usage.get("hasTask")) if norm(t)}
        cats_set = {norm(c) for c in as_list(usage.get("hasTaskCategory")) if norm(c)}
        concepts_set = tasks_set | cats_set

        task_counter.update(tasks_set)
        cat_counter.update(cats_set)
        by_dataset[ds] = {
            "tasks_set": tasks_set,
            "categories_set": cats_set,
            "concepts_set": concepts_set,
            "tasks": len(tasks_set),
            "categories": len(cats_set),
            "concepts": len(concepts_set),
        }

    summary = {
        "dataset_count": len(by_dataset),
        "task_links": sum(task_counter.values()),
        "category_links": sum(cat_counter.values()),
        "unique_tasks": len(task_counter),
        "unique_categories": len(cat_counter),
    }
    return summary, by_dataset, task_counter, cat_counter

json_summary, json_by_ds, json_tasks, json_cats = json_stats(parsed)
json_summary

{'dataset_count': 901045,
 'task_links': 13595,
 'category_links': 131594,
 'unique_tasks': 74,
 'unique_categories': 57}

In [4]:
def load_ttl_graph(ttl_dirs):
    g = Graph()
    files = []
    for d in ttl_dirs:
        files.extend(sorted(d.glob("*.ttl")))
    for fp in files:
        g.parse(fp, format="turtle")
    return g, files

g, ttl_files = load_ttl_graph(TTL_DIRS)
{"ttl_files": len(ttl_files), "ttl_triples": len(g)}

{'ttl_files': 10, 'ttl_triples': 22516428}

In [5]:
# Test d'unicité basé sur la définition des URI (pas sur le nombre d'occurrences dans les triples)
from rdflib import Graph
from rdflib.term import URIRef
from rdflib.namespace import RDF
from collections import defaultdict
from pathlib import Path

def defined_uris(graph):
    typed_subjects = {s for s in graph.subjects(RDF.type, None) if isinstance(s, URIRef)}
    if typed_subjects:
        return {str(s) for s in typed_subjects}, "rdf:type"
    subjects = {s for s in graph.subjects() if isinstance(s, URIRef)}
    return {str(s) for s in subjects}, "subject"

def check_uri_definition_uniqueness(ttl_files, global_graph):
    # Définitions sur le graph global déjà chargé
    global_defs, global_mode = defined_uris(global_graph)

    # Détection des URI définies dans plusieurs fichiers
    uri_to_files = defaultdict(set)
    file_modes = {}
    for fp in ttl_files:
        fg = Graph()
        fg.parse(fp, format="turtle")
        file_defs, mode = defined_uris(fg)
        file_modes[Path(fp).name] = mode
        for uri in file_defs:
            uri_to_files[uri].add(Path(fp).name)

    multi_file_defs = {
        uri: sorted(files)
        for uri, files in uri_to_files.items()
        if len(files) > 1
    }

    return {
        "definition_mode_global": global_mode,
        "defined_uri_count_global": len(global_defs),
        "defined_uri_count_in_files": len(uri_to_files),
        "multi_file_definition_count": len(multi_file_defs),
        "all_definitions_unique_across_files": len(multi_file_defs) == 0,
        "sample_multi_file_definitions": dict(list(multi_file_defs.items())[:10]),
        "file_definition_modes": file_modes,
    }

uri_definition_report = check_uri_definition_uniqueness(ttl_files, g)
uri_definition_report

{'definition_mode_global': 'rdf:type',
 'defined_uri_count_global': 1974141,
 'defined_uri_count_in_files': 1974141,
 'multi_file_definition_count': 31273,
 'all_definitions_unique_across_files': False,
 'sample_multi_file_definitions': {'http://example.org/mluo/data#creator/khursani8': ['output_batch_1.ttl',
   'output_batch_10.ttl',
   'output_batch_2.ttl',
   'output_batch_3.ttl',
   'output_batch_4.ttl',
   'output_batch_7.ttl',
   'output_batch_9.ttl'],
  'http://example.org/mluo/data#creator/weege007': ['output_batch_1.ttl',
   'output_batch_2.ttl',
   'output_batch_3.ttl',
   'output_batch_4.ttl'],
  'http://example.org/mluo/data#creator/cnut1648': ['output_batch_1.ttl',
   'output_batch_2.ttl'],
  'http://example.org/mluo/data#creator/kraina': ['output_batch_1.ttl',
   'output_batch_2.ttl'],
  'http://example.org/mluo/data#creator/iamjinchen': ['output_batch_1.ttl',
   'output_batch_3.ttl'],
  'http://example.org/mluo/data#usage/720c626605346c52': ['output_batch_1.ttl',
   'out

In [ ]:
SPARQL = {
    "datasets": """
        PREFIX dcat: <http://www.w3.org/ns/dcat#>
        SELECT (COUNT(DISTINCT ?d) AS ?n) WHERE { ?d a dcat:Dataset . }
    """,
    "task_links": """
        PREFIX mluo: <http://example.org/mluo/ontology#>
        SELECT (COUNT(*) AS ?n) WHERE { ?d mluo:hasUsage ?u . ?u mluo:hasTask ?t . }
    """,
    "category_links": """
        PREFIX mluo: <http://example.org/mluo/ontology#>
        SELECT (COUNT(*) AS ?n) WHERE { ?d mluo:hasUsage ?u . ?u mluo:hasTaskCategory ?c . }
    """,
    "per_dataset": """
        PREFIX mluo: <http://example.org/mluo/ontology#>
        PREFIX dct:  <http://purl.org/dc/terms/>
        SELECT ?id (COUNT(DISTINCT ?t) AS ?tasks) (COUNT(DISTINCT ?c) AS ?cats)
        WHERE {
          ?d dct:identifier ?id .
          OPTIONAL { ?d mluo:hasUsage ?u1 . ?u1 mluo:hasTask ?t . }
          OPTIONAL { ?d mluo:hasUsage ?u2 . ?u2 mluo:hasTaskCategory ?c . }
        } GROUP BY ?id
    """,
}

list(SPARQL.keys())

['datasets', 'task_links', 'category_links', 'per_dataset']

In [7]:
def q_count(graph, key):
    row = next(iter(graph.query(SPARQL[key])))
    return int(row[0].toPython())

def canonical(uri):
    s = str(uri)
    if s.startswith(str(MLUO_TH)):
        return norm(s.split("#", 1)[-1])
    if s.startswith(str(DATA_NS) + "task/"):
        return norm(s.split("/task/", 1)[-1].replace("_", "-"))
    if s.startswith(str(DATA_NS) + "task_category/"):
        return norm(s.split("/task_category/", 1)[-1].replace("_", "-"))
    return norm(s)

ttl_tasks = Counter()
ttl_cats = Counter()
ttl_by_ds = {}

for dataset_uri, dataset_id in g.subject_objects(DCTERMS.identifier):
    ds = str(dataset_id)
    tasks_set = set()
    cats_set = set()

    for usage in g.objects(dataset_uri, MLUO.hasUsage):
        for task in g.objects(usage, MLUO.hasTask):
            task_name = canonical(task)
            if task_name:
                tasks_set.add(task_name)
                ttl_tasks[task_name] += 1

        for cat in g.objects(usage, MLUO.hasTaskCategory):
            cat_name = canonical(cat)
            if cat_name:
                cats_set.add(cat_name)
                ttl_cats[cat_name] += 1

    ttl_by_ds[ds] = {
        "tasks_set": tasks_set,
        "categories_set": cats_set,
        "concepts_set": tasks_set | cats_set,
        "tasks": len(tasks_set),
        "categories": len(cats_set),
        "concepts": len(tasks_set | cats_set),
    }

ttl_summary = {
    "dataset_count": q_count(g, "datasets"),
    "task_links": q_count(g, "task_links"),
    "category_links": q_count(g, "category_links"),
    "unique_tasks": len(ttl_tasks),
    "unique_categories": len(ttl_cats),
}
ttl_summary

{'dataset_count': 901045,
 'task_links': 13719,
 'category_links': 131600,
 'unique_tasks': 74,
 'unique_categories': 57}

In [8]:
def compare_summary(a, b):
    out = {}
    for k in sorted(set(a) | set(b)):
        av = a.get(k, 0)
        bv = b.get(k, 0)
        out[k] = {"json": av, "ttl": bv, "delta": bv - av}
    return out

summary_cmp = compare_summary(json_summary, ttl_summary)
summary_cmp

{'category_links': {'json': 131594, 'ttl': 131600, 'delta': 6},
 'dataset_count': {'json': 901045, 'ttl': 901045, 'delta': 0},
 'task_links': {'json': 13595, 'ttl': 13719, 'delta': 124},
 'unique_categories': {'json': 57, 'ttl': 57, 'delta': 0},
 'unique_tasks': {'json': 74, 'ttl': 74, 'delta': 0}}

In [9]:
def dataset_mismatches(expected, actual):
    rows = []
    for ds, e in expected.items():
        a = actual.get(
            ds,
            {
                "tasks_set": set(),
                "categories_set": set(),
                "concepts_set": set(),
                "tasks": 0,
                "categories": 0,
                "concepts": 0,
            },
        )

        if e["concepts_set"] != a["concepts_set"]:
            missing_in_ttl = sorted(e["concepts_set"] - a["concepts_set"])
            extra_in_ttl = sorted(a["concepts_set"] - e["concepts_set"])
            rows.append({
                "dataset": ds,
                "json_tasks": e["tasks"],
                "ttl_tasks": a["tasks"],
                "json_categories": e["categories"],
                "ttl_categories": a["categories"],
                "json_concepts": e["concepts"],
                "ttl_concepts": a["concepts"],
                "missing_in_ttl": missing_in_ttl,
                "extra_in_ttl": extra_in_ttl,
            })

    rows.sort(
        key=lambda r: (len(r["missing_in_ttl"]) + len(r["extra_in_ttl"]), r["dataset"]),
        reverse=True,
    )
    return rows

mismatches = dataset_mismatches(json_by_ds, ttl_by_ds)
{"mismatch_count": len(mismatches), "sample": mismatches[:10]}

{'mismatch_count': 0, 'sample': []}

In [10]:
def deltas_by_sign(a_counter, b_counter, n=10):
    keys = set(a_counter) | set(b_counter)
    rows = []
    for k in keys:
        av = a_counter.get(k, 0)
        bv = b_counter.get(k, 0)
        d = bv - av
        if d != 0:
            rows.append((k, av, bv, d))

    negatives = sorted((r for r in rows if r[3] < 0), key=lambda x: abs(x[3]), reverse=True)[:n]
    positives = sorted((r for r in rows if r[3] > 0), key=lambda x: abs(x[3]), reverse=True)[:n]
    net = sum(r[3] for r in rows)
    return negatives, positives, net, len(rows)

task_neg, task_pos, task_net, task_delta_count = deltas_by_sign(json_tasks, ttl_tasks, n=10)
cat_neg, cat_pos, cat_net, cat_delta_count = deltas_by_sign(json_cats, ttl_cats, n=20)

verdict = "OK" if not mismatches and task_delta_count == 0 and cat_delta_count == 0 else "ECARTS A ANALYSER"
{
    "verdict": verdict,
    "mismatch_count": len(mismatches),
    "task_delta_count": task_delta_count,
    "category_delta_count": cat_delta_count,
    "task_delta_net": task_net,
    "category_delta_net": cat_net,
    "top_task_negatives": task_neg,
    "top_task_positives": task_pos,
    "top_category_negatives": cat_neg,
    "top_category_positives": cat_pos,
}

{'verdict': 'ECARTS A ANALYSER',
 'mismatch_count': 0,
 'task_delta_count': 2,
 'category_delta_count': 2,
 'task_delta_net': 124,
 'category_delta_net': 6,
 'top_task_negatives': [],
 'top_task_positives': [('visual-question-answering', 85, 204, 119),
  ('document-question-answering', 25, 30, 5)],
 'top_category_negatives': [],
 'top_category_positives': [('visual-question-answering', 1546, 1551, 5),
  ('document-question-answering', 162, 163, 1)]}